# ДЗ 5. CTE і проєктування ознак для ML: Olist Brazilian E-Commerce

Дані: публічний датасет [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle `olistbr/brazilian-ecommerce`), замовлення бразильського маркетплейсу за 2016-2018 роки. Завантажую 5 CSV у raw-таблиці, переношу їх у typed-схему з PK, FK і CHECK, будую CTE-pipeline ознак для кожного customer, рахую referral-ланцюжок через recursive CTE, кешую ознаки в materialized view і порівнюю плани через `EXPLAIN (ANALYZE, BUFFERS)`.

Префікси таблиць:

| Префікс | Що це |
|---|---|
| `olist_*_raw` | сирі CSV як є, проміжний шар |
| `olist_*` | typed-схема з constraints, усі завдання працюють з нею |
| `hw5_*` | синтетична таблиця `hw5_referrals` (в Olist немає рефералів) |
| `mv_*` | materialized view з ознаками |

PostgreSQL 16 запускається прямо в runtime через пакет `pixeltable-pgserver`: оригінальний `pgserver` на поточній версії Python у Colab не встановлюється, а цей форк має той самий API.

Кожен крок має короткі блоки: **Що робимо** → **Навіщо** → після результату **Що вийшло**.

## Завдання 1. Setup і typed-схема Olist

**Що робимо:** ставимо пакети, запускаємо PostgreSQL і перевіряємо підключення.

**Навіщо:** перша кодова секція має запускати pgserver і показувати версію PostgreSQL. `kagglehub` качає датасет, `jupysql` дає клітинки `%%sql`.

In [1]:
%pip install -q "pixeltable-pgserver==0.4.0" psycopg2-binary sqlalchemy pandas kagglehub jupysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re

import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

# PostgreSQL усередині runtime; файли кластера лежать у /tmp/hw5_pg, поки живе runtime
pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
# Форк за замовчуванням дає URI під psycopg3; явно беремо psycopg2, який ставить перша клітинка
engine = create_engine(pg.get_uri(driver='psycopg2'), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.10 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [3]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)  # jupysql на Python 3.12+ сипле їх зі своїх docstring

%load_ext sql
%config SqlMagic.displaylimit = 50
%config SqlMagic.displaycon = False
%sql engine

**Що вийшло:** PostgreSQL 16.10 запустився всередині runtime, підключення через SQLAlchemy і `%%sql` працює.

### 1.1. Дані Olist з Kaggle через kagglehub

**Що робимо:** `kagglehub.dataset_download('olistbr/brazilian-ecommerce')` качає архів датасету і розпаковує його в кеш kagglehub.

**Навіщо:** ментор повторить роботу без ручного пошуку файлів. Датасет публічний, тому Kaggle credentials не потрібні: перевірено на машині без `~/.kaggle/kaggle.json` і без змінних `KAGGLE_*`. CSV не лежать у репозиторії, лише в кеші поза ним. Шлях до кешу містить домашню папку, тому друкую тільки імена файлів і розміри.

In [4]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')

for file_name in sorted(os.listdir(dataset_dir)):
    size_mb = os.path.getsize(os.path.join(dataset_dir, file_name)) / 1e6
    print(f'{file_name:42} {size_mb:6.1f} MB')

olist_customers_dataset.csv                   9.0 MB
olist_geolocation_dataset.csv                61.3 MB
olist_order_items_dataset.csv                15.4 MB
olist_order_payments_dataset.csv              5.8 MB
olist_order_reviews_dataset.csv              14.5 MB
olist_orders_dataset.csv                     17.7 MB
olist_products_dataset.csv                    2.4 MB
olist_sellers_dataset.csv                     0.2 MB
product_category_name_translation.csv         0.0 MB


**Що вийшло:** датасет завантажився без Kaggle credentials, в архіві 9 CSV. Беремо 5: customers, orders, order_items, products, sellers. Платежі, відгуки, геолокація і переклад категорій для цього ДЗ не потрібні.

### 1.2. Raw-шар: CSV → `olist_*_raw`

**Що робимо:** читаємо 5 CSV через `pd.read_csv` і пишемо кожен у свою raw-таблицю через `to_sql(..., if_exists='replace')`.

**Навіщо:** raw-шар повторює CSV один до одного, без constraints. Це проміжний шар: типи, ключі й перевірки накладаємо далі в typed-схемі одним SQL-запитом. `if_exists='replace'` робить повторний запуск безпечним.

In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


Які типи pandas дав колонкам у raw-таблицях: таймстемпи лишилися текстом, числові колонки продуктів стали `double precision` через пропуски.

In [6]:
%%sql
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('olist_orders_raw', 'olist_products_raw')
ORDER BY table_name, ordinal_position;

17 rows affected.

table_name,column_name,data_type
olist_orders_raw,order_id,text
olist_orders_raw,customer_id,text
olist_orders_raw,order_status,text
olist_orders_raw,order_purchase_timestamp,text
olist_orders_raw,order_approved_at,text
olist_orders_raw,order_delivered_carrier_date,text
olist_orders_raw,order_delivered_customer_date,text
olist_orders_raw,order_estimated_delivery_date,text
olist_products_raw,product_id,text
olist_products_raw,product_category_name,text


**Що вийшло:** 5 raw-таблиць: customers 99 441 × 5, orders 99 441 × 8, order_items 112 650 × 7, products 32 951 × 9, sellers 3 095 × 4. Таймстемпи в `olist_orders_raw` мають тип `text`, тож як дати їх без касту не порівняти. Сім числових колонок `olist_products_raw` pandas прочитав як `double precision`, бо в них є пропуски (NaN), і записав ці пропуски як SQL `NULL`. У typed-схемі `::INTEGER` повертає колонкам цілий тип.

### 1.3. Typed-схема

**Що робимо:** створюємо 5 typed-таблиць з DDL умови: PK, FK, `NOT NULL`, `CHECK` на статус замовлення і невід'ємні розміри та ціни. У typed-схемі назви нормалізовані: `product_name_length`, `product_description_length`.

**Навіщо:** усі наступні завдання працюють тільки з typed-таблицями: тут дата вже `TIMESTAMP`, ціна `NUMERIC`, а FK гарантує, що кожен item посилається на справжнє замовлення, товар і продавця. `DROP ... CASCADE` на початку робить Restart & Run All повторюваним: CASCADE знесе й MV та FK з `hw5_referrals`, обидва створюються нижче заново.

In [7]:
%%sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

-- Які обмеження створив Postgres (NOT NULL сюди не входять)
SELECT conrelid::regclass AS table_name,
       CASE contype WHEN 'p' THEN 'PRIMARY KEY' WHEN 'f' THEN 'FOREIGN KEY' WHEN 'c' THEN 'CHECK' END AS constraint_type,
       COUNT(*) AS n_constraints
FROM pg_constraint
WHERE conrelid IN ('olist_customers'::regclass, 'olist_orders'::regclass, 'olist_products'::regclass,
                   'olist_sellers'::regclass, 'olist_order_items'::regclass)
GROUP BY conrelid, contype
ORDER BY conrelid::regclass::TEXT, constraint_type;

10 rows affected.

table_name,constraint_type,n_constraints
olist_customers,PRIMARY KEY,1
olist_order_items,CHECK,2
olist_order_items,FOREIGN KEY,3
olist_order_items,PRIMARY KEY,1
olist_orders,CHECK,1
olist_orders,FOREIGN KEY,1
olist_orders,PRIMARY KEY,1
olist_products,CHECK,7
olist_products,PRIMARY KEY,1
olist_sellers,PRIMARY KEY,1


**Що вийшло:** Postgres створив 5 PRIMARY KEY, 4 FOREIGN KEY (orders → customers; order_items → orders, products, sellers) і 10 CHECK: статус замовлення, 7 невід'ємних характеристик товару, `price` і `freight_value`.

### 1.4. Перенесення raw → typed: `INSERT ... SELECT`

**Що робимо:** переносимо дані за шаблоном умови, у порядку FK: customers → orders → products → sellers → order_items. Касти `::TIMESTAMP`, `::INTEGER`, `::NUMERIC(12, 2)` перетворюють сирі значення на типи схеми; з raw читаємо фактичні назви колонок `product_name_lenght` і `product_description_lenght` (з помилкою, як у CSV).

**Навіщо:** якщо хоч одне значення не пройде каст, `CHECK` або FK, `INSERT` впаде, і ми побачимо проблему одразу, а не в середині pipeline. Після вставки `ANALYZE` оновлює статистику таблиць, щоб планувальник у Завданні 5 знав справжню кількість рядків.

In [8]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

-- Свіжа статистика для планувальника: оцінки рядків у EXPLAIN (Завдання 5) спираються на неї
ANALYZE olist_customers, olist_orders, olist_products, olist_sellers, olist_order_items;

-- Після перенесення дати вже TIMESTAMP, порожні значення стали NULL
SELECT order_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_customer_date
FROM olist_orders
ORDER BY order_purchase_timestamp, order_id
LIMIT 5;

99441 rows affected.

99441 rows affected.

32951 rows affected.

3095 rows affected.

112650 rows affected.

5 rows affected.

order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date
2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,None
e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,None
809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,None
bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-09 07:47:38
71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02 22:07:52,2016-10-06 15:50:56,None


**Що вийшло:** усі 5 `INSERT` пройшли без помилок: 99 441 customers, 99 441 orders, 32 951 products, 3 095 sellers, 112 650 order_items. Жоден каст, `CHECK` чи FK не відкинув рядок. Порожні дати стали `NULL` (у виводі `None`). У перших замовленнях немає `order_delivered_customer_date`, бо їх не доставили (`shipped`, `canceled`).

### 1.5. Smoke-test: raw vs typed

**Що робимо:** smoke-test кількості рядків з умови, розширений до порівняння: для кожної таблиці `n_raw`, `n_typed` і різниця.

**Навіщо:** якщо typed-таблиця має менше рядків, ніж raw, частина даних загубилася при перенесенні, і це треба пояснити.

In [9]:
%%sql
WITH counts AS (
    SELECT 'customers' AS table_name,
           (SELECT COUNT(*) FROM olist_customers_raw) AS n_raw,
           (SELECT COUNT(*) FROM olist_customers) AS n_typed
    UNION ALL
    SELECT 'orders', (SELECT COUNT(*) FROM olist_orders_raw), (SELECT COUNT(*) FROM olist_orders)
    UNION ALL
    SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw), (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL
    SELECT 'products', (SELECT COUNT(*) FROM olist_products_raw), (SELECT COUNT(*) FROM olist_products)
    UNION ALL
    SELECT 'sellers', (SELECT COUNT(*) FROM olist_sellers_raw), (SELECT COUNT(*) FROM olist_sellers)
)
SELECT table_name, n_raw, n_typed, n_raw - n_typed AS diff
FROM counts
ORDER BY table_name;

5 rows affected.

table_name,n_raw,n_typed,diff
customers,99441,99441,0
order_items,112650,112650,0
orders,99441,99441,0
products,32951,32951,0
sellers,3095,3095,0


**Що вийшло:** `diff = 0` у всіх п'яти таблицях. Typed-схема має рівно стільки рядків, скільки raw, тож при перенесенні нічого не загубилося.

### 1.6. Перевірка зерна: `customer_id` vs `customer_unique_id`

**Що робимо:** рахуємо, скільки різних `customer_id` і `customer_unique_id` є в customers, і скільки ключів кожного типу мають більше одного замовлення.

**Навіщо:** усі ознаки в pipeline рахуються per `customer_id`, як у шаблоні умови. Перед тим, як рахувати `frequency` чи rolling AOV, треба знати, що саме означає `customer_id` в Olist.

In [10]:
%%sql
WITH orders_per_customer_id AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY customer_id
),
orders_per_unique_id AS (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT 'customer_id' AS grain,
       (SELECT COUNT(DISTINCT customer_id) FROM olist_customers) AS n_distinct_in_customers,
       COUNT(*) FILTER (WHERE n_orders > 1) AS n_keys_with_2plus_orders,
       MAX(n_orders) AS max_orders_per_key
FROM orders_per_customer_id
UNION ALL
SELECT 'customer_unique_id',
       (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers),
       COUNT(*) FILTER (WHERE n_orders > 1),
       MAX(n_orders)
FROM orders_per_unique_id;

2 rows affected.

grain,n_distinct_in_customers,n_keys_with_2plus_orders,max_orders_per_key
customer_id,99441,0,1
customer_unique_id,96096,2997,17


**Що вийшло:** в Olist `customer_id` видається на кожне замовлення. Різних `customer_id` 99 441, і жоден не має більше одного замовлення (max = 1). Реальну людину позначає `customer_unique_id`: таких 96 096, з них 2 997 зробили 2+ замовлення, рекордсмен зробив 17.

Що це означає для ознак на зерні `customer_id`: `frequency` завжди 1, `cancellation_rate` буває лише 0 або 1, а rolling 30-day AOV дорівнює сумі єдиного замовлення (перевірка на всіх рядках у 4.1). Pipeline технічно коректний і відповідає умові, тому код лишаю за шаблоном. Для production ML я б агрегував по людині: у stage1 додати `JOIN olist_customers`, брати `c.customer_unique_id` і групувати по ньому в усіх stages.

## Завдання 2. CTE-chain pipeline ознак

**Що робимо:** будуємо pipeline з 8 named stages (Stage 0-7). Кожен stage робить одну трансформацію і має коментар `-- Stage N: ...`.

**Навіщо:** ланцюжок CTE читається як конвеєр: кожну станцію можна перевірити окремо (це зроблено нижче, у перевірці кількості рядків). `reference_date` береться з даних (`MAX(order_purchase_timestamp)::DATE + 1`), а не `CURRENT_DATE`: датасет історичний, і з `CURRENT_DATE` `recency_days` змінювався б щодня.

| Stage | Що робить | Яку feature дає |
|---|---|---|
| `reference_date` (0) | дата «зараз» для датасету: останнє замовлення + 1 день | база для `recency_days` |
| `stage1_orders_total` (1) | `LEFT JOIN` orders → items, сума `price + freight_value` на замовлення | `order_total`, статус, дата |
| `stage2_rfm` (2) | `GROUP BY customer_id` над stage1 | `recency_days`, `frequency`, `monetary`, `first_order_date` |
| `stage3_cohort` (3) | `DATE_TRUNC('quarter', first_order_date)` | `cohort_quarter` |
| `stage4_rolling_aov` (4) | window `AVG ... RANGE 30 days PRECEDING` + `ROW_NUMBER` для останнього замовлення | `avg_order_value_30d` |
| `stage5_cancel_rate` (5) | частка замовлень зі статусом `canceled` | `cancellation_rate` |
| `stage6_category_revenue` (6) | orders → items → products, виручка per customer × категорія | `category_revenue` |
| `stage7_preferred_category` (7) | `ROW_NUMBER() ... WHERE rn = 1` над stage6 | `preferred_category` |

Фінальний `SELECT` бере всіх customers і через `LEFT JOIN` додає ознаки; для customers без замовлень чи items спрацьовують `COALESCE` (`9999`, `0`, `'(none)'`).

In [11]:
%%sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

20 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


**Що вийшло:** у top-20 за `monetary` замовлення від 3 826.80 до 13 664.08, лідер з RJ і категорією `telefonia_fixa`. У всіх 20 рядках `frequency = 1`, а `avg_order_value_30d_recent` і `preferred_category_revenue` збігаються з `monetary`. Це та сама пастка зерна з 1.6. `recency_days` від 76 до 621, когорти від `2017-01-01` до `2018-07-01`.

У результаті видно ще два нюанси шаблону. `monetary` рахує замовлення будь-якого статусу, тому customer з GO з `cancellation_rate = 1.0000` (скасоване замовлення на 4 809.44) теж потрапив у топ. Один customer має категорію `(unknown)`: у його товару немає `product_category_name`, і `COALESCE` у stage6 підставив заглушку. Категорії записані португальською, напр. `telefonia_fixa` означає стаціонарну телефонію, `eletroportateis` дрібну побутову техніку.

### 2.1. Перевірка кожного stage окремо

**Що робимо:** той самий ланцюжок CTE, але фінальний `SELECT` повертає кількість рядків і різних `customer_id` у кожному stage, поруч із джерелами.

**Навіщо:** так видно, що зерно кожного етапу правильне. У stage1 рядок означає замовлення, у stage2/3/5 customer із замовленнями, у stage4 знову замовлення (window не зменшує кількість рядків), у stage7 customer з items.

In [12]:
%%sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT 1 AS step, 'olist_orders (джерело)' AS stage, COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_customers
FROM olist_orders
UNION ALL
SELECT 2, 'olist_customers (джерело)', COUNT(*), COUNT(DISTINCT customer_id) FROM olist_customers
UNION ALL
SELECT 3, 'reference_date', COUNT(*), NULL FROM reference_date
UNION ALL
SELECT 4, 'stage1_orders_total', COUNT(*), COUNT(DISTINCT customer_id) FROM stage1_orders_total
UNION ALL
SELECT 5, 'stage2_rfm', COUNT(*), COUNT(DISTINCT customer_id) FROM stage2_rfm
UNION ALL
SELECT 6, 'stage3_cohort', COUNT(*), COUNT(DISTINCT customer_id) FROM stage3_cohort
UNION ALL
SELECT 7, 'stage4_rolling_aov', COUNT(*), COUNT(DISTINCT customer_id) FROM stage4_rolling_aov
UNION ALL
SELECT 8, 'stage4_rolling_aov, rn_recent = 1', COUNT(*), COUNT(DISTINCT customer_id)
FROM stage4_rolling_aov WHERE rn_recent = 1
UNION ALL
SELECT 9, 'stage5_cancel_rate', COUNT(*), COUNT(DISTINCT customer_id) FROM stage5_cancel_rate
UNION ALL
SELECT 10, 'stage6_category_revenue', COUNT(*), COUNT(DISTINCT customer_id) FROM stage6_category_revenue
UNION ALL
SELECT 11, 'stage7_preferred_category', COUNT(*), COUNT(DISTINCT customer_id) FROM stage7_preferred_category
ORDER BY step;

11 rows affected.

step,stage,n_rows,n_customers
1,olist_orders (джерело),99441,99441
2,olist_customers (джерело),99441,99441
3,reference_date,1,None
4,stage1_orders_total,99441,99441
5,stage2_rfm,99441,99441
6,stage3_cohort,99441,99441
7,stage4_rolling_aov,99441,99441
8,"stage4_rolling_aov, rn_recent = 1",99441,99441
9,stage5_cancel_rate,99441,99441
10,stage6_category_revenue,99470,98666


**Що вийшло:** зерно кожного етапу таке, як задумано.
- stage1: 99 441 рядків, по одному на замовлення (і на `customer_id`, бо в Olist це одне й те саме).
- stage2, stage3, stage5: 99 441 customers із замовленнями.
- stage4 зберігає всі 99 441 замовлення, а `rn_recent = 1` лишає рівно одне на customer.
- stage6: 99 470 пар customer × категорія для 98 666 customers. Рядків на 804 більше, бо частина замовлень має товари з кількох категорій.
- stage7: 98 666 рядків, по одному на customer з items.

Різниця 99 441 − 98 666 = 775 customers, у чиєму замовленні немає жодного item. `LEFT JOIN` у stage1 і `COALESCE` лишають їх у фінальному результаті з `preferred_category = '(none)'` і `monetary = 0`.

## Завдання 3. Recursive CTE для referral chain

**Що робимо:** створюємо синтетичну `hw5_referrals` за шаблоном умови: перші 10 customers за `customer_id` з'єднуємо в ланцюжок, кожен запрошує наступного (9 пар).

**Навіщо:** в Olist немає таблиці рефералів, а recursive CTE потрібен граф. FK на `olist_customers` гарантує, що в графі лише справжні customers, `PRIMARY KEY (referred_customer_id)` не дає запросити одного customer двічі, а `CHECK` забороняє запросити самого себе.

У виводі явні колонки без `referred_at`: там `NOW()`, і значення мінялося б з кожним запуском.

In [13]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT referrer_customer_id, referred_customer_id
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;

9 rows affected.

9 rows affected.

referrer_customer_id,referred_customer_id
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130


**Що вийшло:** 9 пар. Перші 10 customers за `customer_id` утворюють один ланцюжок `00012a2c… → 000161a0… → … → 000598ca…`.

### 3.1. Глибина і шлях у referral-tree

**Що робимо:** recursive CTE з шаблону. Anchor: customers, яких ніхто не запросив (`NOT EXISTS`), з `depth = 1`. Recursive part: від referrer до його referred, `depth + 1`, шлях дописується в масив `path_ids` і рядок `path_text`.

**Навіщо:** recursive CTE обходить граф будь-якої глибини одним запитом. Від циклів захищають два запобіжники: `NOT r.referred_customer_id = ANY(rd.path_ids)` (не заходити у вже відвіданого customer) і `rd.depth < 50` (жорстка межа глибини). Anchor через `NOT EXISTS`, а не `NOT IN`: `NOT IN` дає 0 рядків, якщо в підзапиті трапиться `NULL`.

In [14]:
%%sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;

10 rows affected.

depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


**Що вийшло:** максимальна глибина 10, як і довжина ланцюжка. На `depth = 1` 99 432 customers: усі, кого ніхто не запросив (99 441 − 9 referred). Це корінь ланцюжка `00012a2c…` і ще 99 431 customers без рефералів. Anchor бере кожного, хто не є `referred`, тож кожен такий customer стає окремим «деревом» з одного вузла. На глибинах від 2 до 10 по одному customer, а `sample_path` на depth 10 показує весь шлях від кореня до `000598ca…`. На depth 1 `MAX(path_text)` дає просто найбільший `customer_id` серед коренів.

### 3.2. Що буде з циклом A → B → A

**Що робимо:** не змінюючи `hw5_referrals`, запускаємо ту саму логіку над двома inline-ребрами `('A', 'B'), ('B', 'A')` зі стартом у `'A'`. Спершу з guard на `path_ids`, потім з вбудованим `CYCLE` (PostgreSQL 14+).

**Навіщо:** у реальних даних цикл може з'явитися через помилку імпорту. Треба показати, що запит при цьому зупиняється, а не крутиться вічно.

In [15]:
%%sql
-- Варіант 1: guard на масив відвіданих вузлів + depth guard, як у шаблоні
WITH RECURSIVE edges (referrer_id, referred_id) AS (
    VALUES ('A', 'B'), ('B', 'A')
),
walk AS (
    SELECT
        'A'::TEXT AS customer_id,
        1 AS depth,
        ARRAY['A']::TEXT[] AS path_ids,
        'A'::TEXT AS path_text

    UNION ALL

    SELECT
        e.referred_id,
        w.depth + 1,
        w.path_ids || e.referred_id,
        w.path_text || ' -> ' || e.referred_id
    FROM edges AS e
    JOIN walk AS w
        ON w.customer_id = e.referrer_id
    WHERE NOT e.referred_id = ANY(w.path_ids)
      AND w.depth < 50
)
SELECT depth, customer_id, path_text
FROM walk
ORDER BY depth;

2 rows affected.

depth,customer_id,path_text
1,A,A
2,B,A -> B


In [16]:
%%sql
-- Варіант 2: CYCLE-clause, Postgres сам веде шлях і позначає рядок, що замкнув цикл
WITH RECURSIVE edges (referrer_id, referred_id) AS (
    VALUES ('A', 'B'), ('B', 'A')
),
walk AS (
    SELECT 'A'::TEXT AS customer_id, 1 AS depth

    UNION ALL

    SELECT e.referred_id, w.depth + 1
    FROM edges AS e
    JOIN walk AS w
        ON w.customer_id = e.referrer_id
) CYCLE customer_id SET is_cycle USING path
SELECT depth, customer_id, is_cycle, path::TEXT AS path
FROM walk
ORDER BY depth;

3 rows affected.

depth,customer_id,is_cycle,path
1,A,False,{(A)}
2,B,False,"{(A),(B)}"
3,A,True,"{(A),(B),(A)}"


Нюанс anchor з шаблону: якщо цикл замкнутий повністю, у ньому немає кореня. Скільки коренів знайде `NOT EXISTS` у графі A → B → A:

In [17]:
%%sql
WITH edges (referrer_id, referred_id) AS (
    VALUES ('A', 'B'), ('B', 'A')
),
nodes (customer_id) AS (
    VALUES ('A'), ('B')
)
SELECT COUNT(*) AS n_roots
FROM nodes AS n
WHERE NOT EXISTS (
    SELECT 1
    FROM edges AS e
    WHERE e.referred_id = n.customer_id
);

1 rows affected.

n_roots
0


**Що вийшло:** з guard запит повернув 2 рядки (`A`, `A -> B`) і зупинився. Крок назад у `A` відкинула умова `NOT e.referred_id = ANY(w.path_ids)`, нових рядків 0, тож рекурсія завершилась. З `CYCLE` рядків 3: на `depth = 3` Postgres ще раз видає `A`, ставить `is_cycle = True` і цей шлях далі не розгортає. Без обох захистів кожна ітерація додавала б ще один рядок A ↔ B, і запит крутився б, доки не скінчиться пам'ять або хтось не зробить cancel. Такий варіант я не запускаю.

Нюанс шаблону: у повністю замкнутому циклі `NOT EXISTS` знаходить 0 коренів. Anchor не стартує, і такі customers просто не потрапляють у результат. Запит завершиться без помилки, тому пропуск легко не помітити, і його варто ловити окремою перевіркою.

## Завдання 4. Materialized View з CONCURRENT REFRESH

**Що робимо:** обгортаємо повний pipeline із Завдання 2 у `mv_olist_customer_features`: той самий ланцюжок і фінальний `SELECT`, без коментарів `-- Stage N`, `ORDER BY` і `LIMIT`. Потім створюємо `UNIQUE INDEX` на `customer_id`.

**Навіщо:** VIEW схожий на «рецепт» і перераховується при кожному читанні. MV схожа на «фото» результату: ознаки пораховано один раз, а читання бере готові рядки. `UNIQUE INDEX` дає швидкий lookup за `customer_id` і потрібен для `REFRESH ... CONCURRENTLY`.

In [18]:
%%sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

ANALYZE mv_olist_customer_features;

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname;

99441 rows affected.

1 rows affected.

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)


In [19]:
%%sql
SELECT COUNT(*) AS n_rows,
       (SELECT COUNT(*) FROM olist_customers) AS n_customers
FROM mv_olist_customer_features;

1 rows affected.

n_rows,n_customers
99441,99441


In [20]:
%%sql
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

5 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


**Що вийшло:** у MV 99 441 рядків, стільки ж, скільки customers, бо фінальний `SELECT` іде від `olist_customers` з `LEFT JOIN`. `UNIQUE INDEX` на `customer_id` створився (видно в `pg_indexes`), отже `customer_id` у MV справді унікальний. Top-5 збігається з першими п'ятьма рядками pipeline із Завдання 2.

### 4.1. Профіль ознак з MV

**Що робимо:** одним запитом до MV рахуємо, скільки customers мають `frequency = 1`, rolling AOV, рівний `monetary`, `cancellation_rate` 0 або 1, і скільки не мають жодного item.

**Навіщо:** перевірити на всіх рядках висновок про зерно з 1.6. Читання з MV не перераховує pipeline, тому такий профіль дешевий.

In [21]:
%%sql
SELECT
    COUNT(*) AS n_customers,
    COUNT(*) FILTER (WHERE frequency = 1) AS n_frequency_1,
    COUNT(*) FILTER (WHERE avg_order_value_30d_recent = monetary) AS n_aov_30d_eq_monetary,
    COUNT(*) FILTER (WHERE cancellation_rate = 1) AS n_cancel_rate_1,
    COUNT(*) FILTER (WHERE cancellation_rate > 0 AND cancellation_rate < 1) AS n_cancel_rate_between,
    COUNT(*) FILTER (WHERE preferred_category = '(none)') AS n_without_items,
    COUNT(DISTINCT cohort_quarter) AS n_cohorts,
    MIN(recency_days) AS min_recency_days,
    MAX(recency_days) AS max_recency_days
FROM mv_olist_customer_features;

1 rows affected.

n_customers,n_frequency_1,n_aov_30d_eq_monetary,n_cancel_rate_1,n_cancel_rate_between,n_without_items,n_cohorts,min_recency_days,max_recency_days
99441,99441,99441,625,0,775,10,1,774


**Що вийшло:** на всіх 99 441 рядках `frequency = 1` і `avg_order_value_30d_recent = monetary`. `cancellation_rate = 1` у 625 customers (у кожного одне скасоване замовлення), у решти 0, проміжних значень немає. Без items 775 customers, когорт 10 (квартали), `recency_days` від 1 до 774. Висновок з 1.6 підтвердився на всіх даних: на зерні `customer_id` три ознаки вироджені.

### 4.2. `REFRESH MATERIALIZED VIEW CONCURRENTLY`

**Що робимо:** оновлюємо MV окремою клітинкою, після створення `UNIQUE INDEX`, і рахуємо рядки ще раз.

**Навіщо:** звичайний `REFRESH` блокує читання MV, поки йде перерахунок. `CONCURRENTLY` будує новий результат поруч і підміняє лише змінені рядки, тому читачі весь час бачать стару версію («переклеїти фото, не закриваючи вітрину»). Для порівняння старих і нових рядків Postgres потрібен `UNIQUE INDEX` на колонках MV без `WHERE`.

In [22]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

++
||
++
++

In [23]:
%%sql
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT customer_id) AS n_unique_customers
FROM mv_olist_customer_features;

1 rows affected.

n_rows,n_unique_customers
99441,99441


**Що вийшло:** `REFRESH ... CONCURRENTLY` пройшов без помилки, після нього в MV ті самі 99 441 рядків і 99 441 унікальних `customer_id`. Postgres перерахував увесь pipeline і порівняв новий результат зі старим через `UNIQUE INDEX`. Дані не змінювалися, тож результат вийшов той самий. Інкрементальним це оновлення не є: кожен refresh коштує як повний перерахунок pipeline (≈ 0.9 s за EXPLAIN у 5.2) плюс порівняння.

## Завдання 5. EXPLAIN ANALYZE + Reflection

**Що робимо:** запускаємо `EXPLAIN (ANALYZE, BUFFERS)` для перерахунку pipeline і для читання з MV.

**Навіщо:** `ANALYZE` виконує запит і показує фактичний час кожного вузла плану, `BUFFERS` показує, скільки сторінок прочитано з кешу (`shared hit`) і з диска (`read`), а також тимчасові файли (`temp`), якщо сортування не вмістилося в пам'ять.

Усі плани виконуються однією функцією через SQLAlchemy і друкуються як текст повністю: таблиця jupysql обрізала б довгі рядки. Час між запусками трохи змінюється, а числа в тексті взято з останнього прогону notebook.

In [24]:
plan_summary = {}


def show_plan(name, sql):
    """Виконує EXPLAIN (ANALYZE, BUFFERS), друкує план повністю і запам'ятовує підсумок."""
    with engine.connect() as conn:
        lines = [row[0] for row in conn.execute(text(sql))]
    print('\n'.join(lines))

    plan_text = '\n'.join(lines)
    root_buffers = next((line for line in lines if line.strip().startswith('Buffers:')), '')

    def buffer_count(pattern):
        match = re.search(pattern, root_buffers)
        return int(match.group(1)) if match else 0

    plan_summary[name] = {
        'root_node': lines[0].split('  (')[0].strip(),
        'execution_ms': float(re.search(r'Execution Time: ([\d.]+) ms', plan_text).group(1)),
        'shared_hit': buffer_count(r'shared hit=(\d+)'),
        'shared_read': buffer_count(r'shared (?:hit=\d+ )?read=(\d+)'),
        'temp_written': buffer_count(r'temp (?:read=\d+ )?written=(\d+)'),
    }


# Скільки пам'яті має один Sort / HashAggregate, перш ніж писати тимчасові файли на диск
with engine.connect() as conn:
    print('work_mem =', conn.execute(text('SHOW work_mem')).scalar_one())

work_mem = 4MB


### 5.1. Pipeline: RFM-частина з умови

**Що робимо:** EXPLAIN шаблону з умови: Stage 0-2 і top-10 за `monetary`.

In [25]:
show_plan('pipeline_rfm_top10', """
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

Limit  (cost=33610.98..33611.01 rows=10 width=65) (actual time=199.010..199.012 rows=10 loops=1)
  Buffers: shared hit=105264, temp read=961 written=1696
  ->  Sort  (cost=33610.98..33611.48 rows=200 width=65) (actual time=199.009..199.011 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=105264, temp read=961 written=1696
        ->  Subquery Scan on stage2_rfm  (cost=33601.16..33606.66 rows=200 width=65) (actual time=144.539..189.135 rows=99441 loops=1)
              Buffers: shared hit=105261, temp read=961 written=1696
              ->  HashAggregate  (cost=33601.16..33604.66 rows=200 width=69) (actual time=144.539..184.703 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=105261, temp read=961 writ

**Що вийшло:** 200.7 ms. Знизу вгору:
- `reference_date` рахує `Aggregate` над `Seq Scan on olist_orders`, ≈ 9 ms.
- stage1 робить `Merge Left Join` двох `Index Scan` (`olist_orders_pkey` і `olist_order_items_pkey`) і `GroupAggregate`, ≈ 106 ms. Обидві таблиці приходять уже впорядкованими за `order_id`, тому окремий Sort не потрібен. За це план платить 100 347 `shared hit` на `Index Scan` по orders, майже одну сторінку heap на рядок.
- stage2 рахує `HashAggregate` до ≈ 185 ms, з `Batches: 21  Disk Usage: 7400kB`. 99 441 груп не вмістилися в `work_mem = 4MB`, і агрегат писав тимчасові файли.
- Нагорі `Sort` з `top-N heapsort` (27 kB): для `LIMIT 10` Postgres тримає в пам'яті лише 10 найкращих рядків замість сортування всіх.

### 5.2. Повний pipeline: усі 8 stages

**Що робимо:** EXPLAIN повного pipeline з Завдання 2 (`ORDER BY monetary ... LIMIT 20`). Тіло запиту беру з визначення MV через `pg_get_viewdef`, тож план гарантовано рахує той самий запит, що й MV.

**Навіщо:** RFM-частина не показує window functions і JOIN з products. Щоб сказати, який feature найдорожчий, потрібен план усього ланцюжка.

In [26]:
with engine.connect() as conn:
    mv_body = conn.execute(
        text("SELECT pg_get_viewdef('mv_olist_customer_features'::regclass, true)")
    ).scalar_one().rstrip().rstrip(';')

show_plan('pipeline_full_top20', f"""
EXPLAIN (ANALYZE, BUFFERS)
{mv_body}
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 20
""")

Limit  (cost=116351.89..116351.94 rows=20 width=150) (actual time=887.335..887.515 rows=20 loops=1)
  Buffers: shared hit=111711, temp read=8396 written=8545
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27069.70 rows=99441 width=103) (actual time=0.028..107.156 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24484.61 rows=112650 width=95) (actual time=0.019..65.531 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12563.59 rows=99441 width=83) (actual time=0.010..36.986 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.006..8.183 rows=112650 loops=1)
               

**Що вийшло:** 893.1 ms і `temp written = 8545` блоків по 8 kB, тобто ≈ 67 MB тимчасових файлів. Час кожної гілки плану (`actual time` вузла включає дочірні вузли):

| Stage | Ключові вузли | Час гілки |
|---|---|---|
| stage1, CTE (рахується один раз для stage2, stage4 і stage5) | `Merge Left Join` + `GroupAggregate` | ≈ 107 ms |
| stage4, rolling AOV | `Sort` (`external merge  Disk: 9448kB`) → `WindowAgg` → `Incremental Sort` → `WindowAgg` | ≈ 280 ms, з них ≈ 139 ms займає `CTE Scan` stage1; власних ≈ 141 ms |
| stage6 + stage7, preferred category | `Parallel Hash Join` orders × items, `Hash Left Join` з products, `Sort` (`external merge`, 4208kB + 3768kB), `GroupAggregate`, `Incremental Sort`, `WindowAgg` | ≈ 173 ms |
| stage2, RFM | `HashAggregate` (`Batches: 21`, 7400kB на диску) | ≈ 103 ms |
| stage5, cancellation rate | `HashAggregate` (`Batches: 21`, 7200kB на диску) | ≈ 80 ms |
| фінальна збірка | 4 `Hash Join` по 99 441 рядків + `top-N heapsort` | решта, до 887 ms |

Найдорожча ознака `preferred_category`: лише вона сама з'єднує три таблиці й сортує результат перед ранжуванням. На другому місці rolling AOV із зовнішнім сортуванням 9.4 MB під вікно. Обидві впираються в `Sort`, який при `work_mem = 4MB` іде на диск.

### 5.3. Lookup з MV за `customer_id`

**Що робимо:** EXPLAIN шаблону з умови: ознаки одного customer з MV.

In [27]:
show_plan('mv_lookup_one_customer', """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
""")

Nested Loop  (cost=0.83..8.96 rows=1 width=92) (actual time=0.032..0.032 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.53 rows=1 width=33) (actual time=0.028..0.029 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11127.92 rows=99441 width=33) (actual time=0.028..0.028 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=92) (actual time=0.002..0.002 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=61 read=1
Planning Time: 0.262 ms
Execution Time: 0.042 ms


**Що вийшло:** 0.042 ms і 8 буферів. `Limit` над `Index Only Scan using olist_customers_pkey` бере першого customer, а `Nested Loop` знаходить його в MV через `Index Scan using idx_mv_olist_customer_features_customer_id` (4 буфери). У плані немає `Seq Scan`, агрегатів чи сортувань, бо ознаки вже пораховані. Це ≈ 21 000 разів швидше за перерахунок pipeline у 5.2 (893 ms).

### 5.4. Top-N з MV та індекс на `monetary`

**Що робимо:** створюємо індекс з умови `(monetary DESC)` і запускаємо EXPLAIN top-10 за `monetary DESC NULLS LAST`.

In [28]:
%%sql
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname;

2 rows affected.

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)
idx_mv_olist_customer_features_monetary,CREATE INDEX idx_mv_olist_customer_features_monetary ON public.mv_olist_customer_features USING btree (monetary DESC)


In [29]:
show_plan('mv_top10_by_monetary', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=4449.01..4450.16 rows=10 width=92) (actual time=7.009..7.316 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=92) (actual time=7.008..7.315 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=92) (actual time=5.925..5.926 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=92) (actual time=0.003..2.671 rows=49720 loops=2)
                    Buffers: shared hit=1600
Planning:
  Buffers: shared hit=20 read=1
Planning Time: 0.158 ms
Execution Time: 7.326 ms


**Що вийшло:** 7.3 ms і 1 636 буферів, але індекс `idx_mv_olist_customer_features_monetary` план не використав. Замість нього `Parallel Seq Scan` усієї MV, `Sort` з `top-N heapsort` у двох процесах і `Gather Merge`. Причина в порядку `NULL`. `DESC` в індексі за замовчуванням означає `DESC NULLS FIRST`, а запит просить `DESC NULLS LAST`. Індекс віддає рядки в іншому порядку, тож сортувати однаково довелося б.

#### Індекс з тим самим порядком `NULLS LAST`

**Що робимо:** додаю індекс `(monetary DESC NULLS LAST)` і повторюю той самий EXPLAIN.

**Навіщо:** перевірити причину з попереднього кроку. Якщо порядок індексу точно збігається з `ORDER BY`, Postgres може взяти перші 10 рядків прямо з індексу, без сортування.

In [30]:
%%sql
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary_desc_nulls_last
    ON mv_olist_customer_features (monetary DESC NULLS LAST);

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname;

3 rows affected.

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)
idx_mv_olist_customer_features_monetary,CREATE INDEX idx_mv_olist_customer_features_monetary ON public.mv_olist_customer_features USING btree (monetary DESC)
idx_mv_olist_customer_features_monetary_desc_nulls_last,CREATE INDEX idx_mv_olist_customer_features_monetary_desc_nulls_last ON public.mv_olist_customer_features USING btree (monetary DESC NULLS LAST)


In [31]:
show_plan('mv_top10_index_nulls_last', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=0.42..1.32 rows=10 width=92) (actual time=0.011..0.018 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary_desc_nulls_last on mv_olist_customer_features  (cost=0.42..8995.85 rows=99441 width=92) (actual time=0.011..0.017 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.126 ms
Execution Time: 0.023 ms


**Що вийшло:** тепер план `Limit` → `Index Scan using idx_mv_olist_customer_features_monetary_desc_nulls_last`. Прочитано 10 рядків і 13 буферів, 0.023 ms замість 7.3 ms (≈ 300 разів швидше), `Sort` зник. Для top-N індекс має збігатися з `ORDER BY` разом із `NULLS FIRST/LAST`. Можна й інакше: писати `ORDER BY monetary DESC`. `monetary` у MV ніколи не `NULL` завдяки `COALESCE`, і тоді підійшов би індекс з умови.

### 5.5. Порівняння сценаріїв

**Що робимо:** зводимо час виконання і буфери кореневого вузла кожного плану в одну таблицю.

In [32]:
pd.DataFrame(plan_summary).T

,root_node,execution_ms,shared_hit,shared_read,temp_written
pipeline_rfm_top10,Limit,200.691,105264,0,1696
pipeline_full_top20,Limit,893.128,111711,0,8545
mv_lookup_one_customer,Nested Loop,0.042,8,0,0
mv_top10_by_monetary,Limit,7.326,1636,0,0
mv_top10_index_nulls_last,Limit,0.023,10,3,0


**Що вийшло:** ключові вузли, час і буфери всіх планів поруч.

| Сценарій | Ключові вузли | Execution Time | Buffers |
|---|---|---|---|
| pipeline, RFM-частина (5.1) | `Merge Left Join`, `GroupAggregate`, `HashAggregate` (21 batch на диску), `top-N heapsort` | 200.7 ms | 105 264 hit, temp written 1 696 |
| повний pipeline (5.2) | те саме + 2 × `WindowAgg`, `external merge` Sort, 4 `Hash Join` | 893.1 ms | 111 711 hit, temp written 8 545 |
| MV lookup одного customer (5.3) | `Nested Loop` + `Index Only Scan` + `Index Scan` | 0.042 ms | 8 hit |
| MV top-10, індекс `(monetary DESC)` (5.4) | `Parallel Seq Scan` + `top-N heapsort` | 7.3 ms | 1 636 hit |
| MV top-10, індекс `DESC NULLS LAST` | `Index Scan` | 0.023 ms | 10 hit + 3 read |

Повний pipeline (893 ms) повільніший за будь-яке читання з MV: приблизно в 120 разів, ніж top-10 через `Seq Scan` (7.3 ms), і приблизно в 39 000 разів, ніж top-10 через `Index Scan` (0.023 ms). Тимчасові файли на диск пише тільки pipeline.

### 5.6. Reflection

**1. Найдорожча ознака.** Повний pipeline у 5.2 виконується 893 ms. Найдорожча гілка `preferred_category` (stage6-7, ≈ 173 ms): `Parallel Hash Join` orders × items, `Hash Left Join` з products, `Sort` з `external merge` на диску, `GroupAggregate`, `Incremental Sort` і `WindowAgg` для `ROW_NUMBER`. Друга rolling AOV (stage4, ≈ 141 ms власного часу): `Sort` на 9.4 MB іде на диск під два `WindowAgg`. `HashAggregate` у stage2 і stage5 теж пишуть на диск (`Batches: 21`), бо `work_mem = 4MB`. Найбільше часу забирають сортування й агрегати, яким не вистачає пам'яті.

**2. Кешувати через MV?** Так. Lookup з MV займає 0.042 ms (`Index Scan`) проти 893 ms перерахунку, top-10 з відповідним індексом 0.023 ms. Olist історичний і не змінюється, тому досить on-demand refresh після завантаження нових даних. Для живого магазину я б оновлював щодоби вночі: кожен `REFRESH CONCURRENTLY` перераховує весь pipeline, а ознаки в днях і кварталах за годину майже не змінюються.

**3. Коли VIEW.** Коли важлива свіжість на момент читання, а читають рідко. Наприклад, ad-hoc аналітика, налагодження stage або звіт по сьогоднішніх замовленнях, де MV завжди відставала б.

**4. Termination.** Recursive CTE зупиняється, коли recursive part повертає 0 нових рядків. У циклі A → B → A без захисту цього ніколи не станеться. Guard `NOT next = ANY(path_ids)` відкидає вже відвіданий вузол (у демо 2 рядки і стоп), `depth < 50` спрацює як запобіжник у будь-якому разі, а `CYCLE ... SET is_cycle` робить те саме вбудовано. Нюанс шаблону: у повністю замкнутому циклі немає кореня, `NOT EXISTS` дає 0 стартів, і такі customers мовчки випадають з результату.

**5. Trade-off.** Головне: recompute-cost проти latency. Сервісу потрібна відповідь за частки мілісекунди, а перерахунок коштує ≈ 0.9 s і ≈ 67 MB тимчасових файлів. Storage-cost малий (99 441 рядків), а freshness для історичних даних майже не важить.